## GeoSpatial data



In [ ]:
import duckdb


In [ ]:
# Connect to DuckDB and load the spatial extension
con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;") ;


In [ ]:
# Define the remote URLs
roads_url = "https://www2.census.gov/geo/tiger/TIGER2024/ROADS/tl_2024_35006_roads.zip"
states_url = "https://www2.census.gov/geo/tiger/TIGER2024/STATE/tl_2024_us_state.zip"
'''  '''
roads_url, states_url

('https://huggingface.co/datasets/rwcitek/DUC-2026/resolve/main/tl_2024_35006_roads.zip',
 'https://www2.census.gov/geo/tiger/TIGER2024/STATE/tl_2024_us_state.zip')

In [ ]:
! curl -L -O {roads_url}


curl -L -O https://www2.census.gov/geo/tiger/TIGER2024/ROADS/tl_2024_35006_roads.zip


In [ ]:
!ls -la

total 2388
drwxr-xr-x 1 root root    4096 Nov 18 12:03 .
drwxr-xr-x 1 root root    4096 Nov 18 11:52 ..
drwxr-xr-x 4 root root    4096 Oct 26 22:18 .config
drwxr-xr-x 1 root root    4096 Oct 26 22:18 sample_data
-rw-r--r-- 1 root root 2425481 Nov 18 12:03 tl_2024_35006_roads.zip


In [ ]:
!unzip -l tl_2024_35006_roads.zip

Archive:  tl_2024_35006_roads.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
        5  2024-09-15 23:23   tl_2024_35006_roads.cpg
  1270425  2024-09-15 23:23   tl_2024_35006_roads.dbf
      165  2024-09-15 23:23   tl_2024_35006_roads.prj
  3745084  2024-09-15 23:23   tl_2024_35006_roads.shp
    21871  2025-05-30 21:04   tl_2024_35006_roads.shp.ea.iso.xml
    50347  2025-05-30 21:04   tl_2024_35006_roads.shp.iso.xml
    78876  2024-09-15 23:23   tl_2024_35006_roads.shx
---------                     -------
  5166773                     7 files


In [ ]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")

# Point directly to the local ZIP file, appending the internal shapefile path
local_zip = "tl_2024_35006_roads.zip"
roads_path = f"/vsizip/{local_zip}/tl_2024_35006_roads.shp"

# Convert it straight to a GeoParquet file
con.execute(f"""
    COPY (
        SELECT * FROM st_read('{roads_path}')
    ) TO 'roads_35006.parquet' (FORMAT PARQUET, COMPRESSION ZSTD);
""")

print("Successfully converted local zipped shapefile to GeoParquet without uncompressing!")

Successfully converted local zipped shapefile to GeoParquet without uncompressing!


In [ ]:
# Extract the base file name
roads_base = roads_url.split("/")[-1].split(".")[0]
states_base = states_url.split("/")[-1].split(".")[0]

roads_base, states_base

('tl_2024_35006_roads', 'tl_2024_us_state')

In [ ]:
!unzip -l {roads_base}.zip {roads_base}.shp


Archive:  tl_2024_35006_roads.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
  3745084  2024-09-15 23:23   tl_2024_35006_roads.shp
---------                     -------
  3745084                     1 file


In [ ]:
# Construct GDAL virtual paths for remote zipped shapefiles
# /vsicurl/ streams over HTTP, and /vsizip/ opens the archive on-the-fly
roads_path = f"/vsizip//vsicurl/{roads_url}/{roads_base}.shp"
states_path = f"/vsizip//vsicurl/{states_url}/{states_base}.shp"

roads_path, states_path


('/vsizip//vsicurl/https://www2.census.gov/geo/tiger/TIGER2024/ROADS/tl_2024_35006_roads.zip/tl_2024_35006_roads.shp',
 '/vsizip//vsicurl/https://www2.census.gov/geo/tiger/TIGER2024/STATE/tl_2024_us_state.zip/tl_2024_us_state.shp')

In [ ]:
roads_url = "https://www2.census.gov/geo/tiger/TIGER2024/ROADS/tl_2024_35006_roads.zip"
states_url = "https://www2.census.gov/geo/tiger/TIGER2024/STATE/tl_2024_us_state.zip"

roads_path = f"/vsizip//vsicurl/{roads_url}/tl_2024_35006_roads.shp"
states_path = f"/vsizip//vsicurl/{states_url}/tl_2024_us_state.shp"


In [ ]:
# Export Roads shapefile to GeoParquet
con.execute(f"""
    COPY (
        SELECT * FROM st_read('{roads_path}')
    ) TO '{roads_base}.parquet' (FORMAT PARQUET, COMPRESSION ZSTD);
""")



In [ ]:
import os
import duckdb

# 1. Force GDAL/libcurl to send a standard browser User-Agent
os.environ['GDAL_HTTP_USERAGENT'] = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'

# Optional: Enable GDAL curl debugging if you want to inspect the handshake and headers in the terminal
# os.environ['CPL_CURL_VERBOSE'] = 'YES'

# 2. Connect and load spatial
con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")

# 3. Test if the Census URL now responds successfully with the custom user-agent
roads_url = "https://www2.census.gov/geo/tiger/TIGER2024/ROADS/tl_2024_35006_roads.zip"
roads_path = f"/vsizip//vsicurl/{roads_url}/tl_2024_35006_roads.shp"

query = f"""
  SELECT count(1)
  FROM st_read('{roads_path}')
"""

con.execute(query)
# print("Successfully streamed directly from Census using a custom user-agent!")



IOException: IO Error: GDAL Error (4): `/vsizip//vsicurl/https://www2.census.gov/geo/tiger/TIGER2024/ROADS/tl_2024_35006_roads.zip/tl_2024_35006_roads.shp' does not exist in the file system, and is not recognized as a supported dataset name.

In [ ]:
# Export Roads shapefile to GeoParquet
con.execute(f"""
    COPY (
        SELECT * FROM st_read('{roads_path}')
    ) TO '{roads_base}.parquet' (FORMAT PARQUET, COMPRESSION ZSTD);
""")



IOException: IO Error: GDAL Error (4): `/vsizip//vsicurl/https://www2.census.gov/geo/tiger/TIGER2024/ROADS/tl_2024_35006_roads.zip/tl_2024_35006_roads.shp' does not exist in the file system, and is not recognized as a supported dataset name.

In [ ]:
# Export States shapefile to GeoParquet
con.execute(f"""
    COPY (
        SELECT * FROM st_read('{states_path}')
    ) TO '{states_base}.parquet' (FORMAT PARQUET, COMPRESSION ZSTD);
""")

!ls -la *.parquet


In [ ]:
duckdb.sql( f'''
  select *
  from './{states_base}.parquet'
  limit 10
  ;
''').df()


In [ ]:
query = f"""
    SELECT
        roads.* EXCLUDE (geom),
        states.STATEFP,
        states.STUSPS,
        states.NAME AS state_name,
        roads.geom AS road_geometry,
        states.geom AS state_geometry
    FROM read_parquet('{roads_base}.parquet') roads
    LEFT JOIN read_parquet('{states_base}.parquet') states
        ON ST_Within(roads.geom, states.geom)
    WHERE roads.FULLNAME LIKE '%I- 25%'
       OR (roads.RTTYP = 'I'
       -- AND roads.FULLNAME LIKE '%25%'
       )
"""
print(query)


In [ ]:
gdf_combined = con.sql(query).df()
gdf_combined
